# tensor-to-device — worked example 1: .to(device) — not in-place, must reassign

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-to-device`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`.to(device)` returns a new tensor on the target device. It does NOT modify the original tensor in place. If you write `x.to('cuda')` without assigning the result, `x` still lives on CPU. The common fix is `x = x.to(device)` — reassigning the variable to the returned tensor.

## Worked solution

**Step 1 — The silent bug.**
Writing `model.weight.to('cuda')` without assignment looks plausible but does nothing to the model parameter — `model.weight` still points to the CPU tensor. This is the #1 device placement bug for PyTorch beginners.

**Step 2 — The correct pattern.**
For individual tensors: `x = x.to(device)`. For models: `model = model.to(device)` or `model.to(device)` (models modify themselves in-place for their submodules, which is an exception).

**Step 3 — `id()` proof.**
After `y = x.to('cpu')` on a CPU tensor, `y is x` is `True` — `.to()` is idempotent: if the tensor is already on the target device, the same object is returned. After `y = x.to('cuda')`, `y is x` is `False` — a new tensor was created.

**Step 4 — Verify device.**
Check `x.device` to confirm placement. It returns a `torch.device` object; compare with `str(x.device)` for string equality.

In [ ]:
import torch as t

t.manual_seed(0)
x = t.randn(3, 4)      # CPU tensor
print('original device:', x.device)   # cpu

# WRONG: does not move x
x.to('cpu')  # result discarded
print('after x.to() (discarded):', x.device)  # still cpu

# CORRECT: reassign
device = 'cuda' if t.cuda.is_available() else 'cpu'
y = x.to(device)
print('y.device:', y.device)          # cpu (or cuda if available)

# Idempotence: to() on already-correct device returns SAME object
z = y.to(device)
print('z is y (idempotent):', z is y)  # True

# Moving to a different device returns a NEW object
if t.cuda.is_available():
    x_cpu = y.to('cpu')
    print('new object after device change:', x_cpu is y)  # False
else:
    print('CUDA not available; skipping cross-device check')